# Análise exploratória e scripts computacionais de notebook.ipynb

- **Repositório:** `analise-de-dados-fundamentos`  
- **Tecnologias:** Pandas, NumPy  
- **Arquivo Original:** `notebook.ipynb`  

---


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
import os
import numpy as np
from PIL import Image, ImageEnhance
from itertools import product

# =============================================
# CONFIGURAÇÕES
# =============================================
input_dir = '/content/drive/MyDrive/Cedulas/Reais'
output_dir = 'reais_todas_combinacoes'
os.makedirs(output_dir, exist_ok=True)

# Resolução retangular para preservar proporção das cédulas
target_width = 64
target_height = 64

# =============================================
# PARÂMETROS DE COMBINAÇÕES
# =============================================
zooms        = [0.9, 1.0, 1.1]    # Zoom (3 opções)
brightnesses = [0.8, 1.0, 1.2]    # Brilho (3 opções)
contrasts    = [0.8, 1.0, 1.2]    # Contraste (3 opções)

# =============================================
# LOOP PRINCIPAL
# =============================================
for nota in os.listdir(input_dir):
    input_path  = os.path.join(input_dir, nota)
    output_path = os.path.join(output_dir, nota)
    os.makedirs(output_path, exist_ok=True)

    for img_name in os.listdir(input_path):
        img_path     = os.path.join(input_path, img_name)
        original_img = Image.open(img_path).convert('RGB').resize((target_width, target_height))

        # Gera todas as combinações de zoom, brilho e contraste
        all_combinations = product(zooms, brightnesses, contrasts)

        for i, (zoom, bright, contrast) in enumerate(all_combinations):
            img = original_img.copy()

            # 1. Zoom
            if zoom != 1.0:
                w, h = img.size
                new_w, new_h = int(w * zoom), int(h * zoom)
                img = img.resize((new_w, new_h))
                left = (new_w - target_width) // 2
                top = (new_h - target_height) // 2
                img = img.crop((left, top, left + target_width, top + target_height))

            # 2. Brilho
            arr = np.array(img).astype(np.float32) * bright
            arr = np.clip(arr, 0, 255).astype(np.uint8)
            img = Image.fromarray(arr)

            # 3. Contraste
            enhancer = ImageEnhance.Contrast(img)
            img = enhancer.enhance(contrast)

            # Salva
            save_name = f"{os.path.splitext(img_name)[0]}_z{zoom}_b{bright}_c{contrast}.jpg"
            save_path = os.path.join(output_path, save_name)
            img.save(save_path)

print(f"✅ Todas as combinações foram geradas em: {output_dir} (resolução {target_width}x{target_height})")


✅ Todas as combinações foram geradas em: reais_todas_combinacoes (resolução 64x64)


In [12]:
import os

# Caminho de entrada
input_dir = 'reais_todas_combinacoes'

# Dicionário para armazenar o número de imagens por valor de cédula
image_count_per_value = {}

# Contador total de imagens
total_images = 0

# Para cada pasta de valor (2, 5, 10 etc.)
for nota in os.listdir(input_dir):
    input_path = os.path.join(input_dir, nota)

    # Verificar se é uma pasta (representando um valor de cédula)
    if os.path.isdir(input_path):
        # Contar as imagens dentro da pasta
        image_count = len([img for img in os.listdir(input_path) if img.endswith(('.jpg', '.jpeg', '.png'))])

        # Armazenar a contagem de imagens para o valor da cédula
        image_count_per_value[nota] = image_count

        # Atualizar o total de imagens
        total_images += image_count

# Exibir a contagem de imagens por valor
print("Contagem de imagens por valor de cédula:")
for nota, count in image_count_per_value.items():
    print(f"Valor: {nota} - Imagens: {count}")

# Exibir o total de imagens
print(f"\nTotal de imagens: {total_images}")


Contagem de imagens por valor de cédula:
Valor: 100 - Imagens: 189
Valor: 200 - Imagens: 108
Valor: 5 - Imagens: 108
Valor: 20 - Imagens: 108
Valor: 2 - Imagens: 162
Valor: 10 - Imagens: 108
Valor: 50 - Imagens: 189

Total de imagens: 972


In [13]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.18.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


Código a ser Melhorado 160x240

Teste

Código Base 64x64

In [ ]:
import os
import glob
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, initializers
import matplotlib.pyplot as plt

# Configurações
dataset_dir = 'reais_todas_combinacoes'
img_size = 64
batch_size = 64
latent_dim = 100
epochs = 5000
save_interval = 50

# Corrigindo o warning do inicializador
weight_init = initializers.RandomNormal(mean=0.0, stddev=0.02, seed=42)

# Função para carregar imagens
def load_dataset(path_glob, img_size, batch_size):
    files = glob.glob(path_glob, recursive=True)

    def preprocess_image(file_path):
        img = Image.open(file_path).convert('RGB')
        img = img.resize((img_size, img_size))
        arr = np.array(img)
        arr = (arr.astype(np.float32) / 127.5) - 1.0
        return arr

    dataset = tf.data.Dataset.from_tensor_slices([preprocess_image(f) for f in files])
    dataset = dataset.shuffle(buffer_size=len(files), reshuffle_each_iteration=True)
    dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return dataset

# Carregar dataset
dataset = load_dataset(os.path.join(dataset_dir, '**/*.jpg'), img_size, batch_size)

# Gerador
def make_generator():
    model = tf.keras.Sequential([
        layers.Dense(4*4*512, use_bias=False, input_shape=(latent_dim,), kernel_initializer=weight_init),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Reshape((4, 4, 512)),

        layers.Conv2DTranspose(256, 5, strides=2, padding='same', use_bias=False, kernel_initializer=weight_init),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(128, 5, strides=2, padding='same', use_bias=False, kernel_initializer=weight_init),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(64, 5, strides=2, padding='same', use_bias=False, kernel_initializer=weight_init),
        layers.BatchNormalization(),
        layers.ReLU(),

        layers.Conv2DTranspose(3, 5, strides=2, padding='same', activation='tanh', kernel_initializer=weight_init)
    ])
    return model

# Discriminador
def make_discriminator():
    model = tf.keras.Sequential([
        layers.Conv2D(64, 5, strides=2, padding='same',
                     input_shape=[img_size, img_size, 3], kernel_initializer=weight_init),
        layers.LeakyReLU(0.2),

        layers.Conv2D(128, 5, strides=2, padding='same', kernel_initializer=weight_init),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2D(256, 5, strides=2, padding='same', kernel_initializer=weight_init),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Conv2D(512, 5, strides=2, padding='same', kernel_initializer=weight_init),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),

        layers.Flatten(),
        layers.Dense(1, kernel_initializer=weight_init)
    ])
    return model

# Inicialização
generator = make_generator()
discriminator = make_discriminator()

# Otimizadores
generator_optimizer = tf.keras.optimizers.Adam(0.0002, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(0.00007, beta_1=0.5)

# Funções de perda
def discriminator_loss(real_output, fake_output):

    real_labels = tf.ones_like(real_output) * tf.random.uniform(shape=tf.shape(real_output), minval=0.7, maxval=1.0)
    fake_labels = tf.zeros_like(fake_output) + tf.random.uniform(shape=tf.shape(fake_output), minval=0.0, maxval=0.3)


    real_loss = tf.keras.losses.binary_crossentropy(real_labels, real_output, from_logits=True)
    fake_loss = tf.keras.losses.binary_crossentropy(fake_labels, fake_output, from_logits=True)
    return tf.reduce_mean(real_loss + fake_loss)


def generator_loss(fake_output):
    return tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(tf.ones_like(fake_output), fake_output, from_logits=True))

# Função corrigida para salvar imagens
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    predictions = (predictions.numpy() + 1) * 127.5  # Converter para numpy e depois para 0-255

    plt.figure(figsize=(4,4))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow(predictions[i].astype(np.uint8))  # Agora funciona pois é numpy array
        plt.axis('off')

    output_dir = 'gan_output_2'
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(os.path.join(output_dir, f'epoch_{epoch:04d}.png'))
    plt.close()

# Semente fixa
seed = tf.random.normal([16, latent_dim])

# Treinamento
@tf.function
def train_step(images):
    noise = tf.random.normal([batch_size, latent_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)

        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss


def train(dataset, epochs):
    for epoch in range(epochs):
        for image_batch in dataset:
            gen_loss, disc_loss = train_step(image_batch)

        print(f'Epoch {epoch+1}/{epochs}, Gen Loss: {gen_loss:.4f}, Disc Loss: {disc_loss:.4f}')

        if (epoch + 1) >= 50 and (epoch + 1) % save_interval == 0:
            generate_and_save_images(generator, epoch + 1, seed)

        # Salvar os modelos nas épocas 3600 e 3800
        model_dir = 'C:\\Users\\Victor\\PyCharmMiscProject'
        if (epoch + 1) in [3600, 3800]:
            generator.save(os.path.join(model_dir, f'generator_epoch_{epoch+1}.h5'))
            discriminator.save(os.path.join(model_dir, f'discriminator_epoch_{epoch+1}.h5'))
            print(f'Modelos salvos na época {epoch+1}')



# Iniciar
print("Iniciando treinamento...")
train(dataset, epochs)
print("Treinamento concluído!")

Iniciando treinamento...
Epoch 1/5000, Gen Loss: 0.5424, Disc Loss: 1.6797
Epoch 2/5000, Gen Loss: 3.2776, Disc Loss: 1.0556
Epoch 3/5000, Gen Loss: 5.8808, Disc Loss: 1.2654
Epoch 4/5000, Gen Loss: 5.9949, Disc Loss: 1.6881
Epoch 5/5000, Gen Loss: 5.7256, Disc Loss: 1.5728
Epoch 6/5000, Gen Loss: 5.4025, Disc Loss: 1.4930
Epoch 7/5000, Gen Loss: 6.1630, Disc Loss: 1.4396
Epoch 8/5000, Gen Loss: 5.6229, Disc Loss: 1.1880
Epoch 9/5000, Gen Loss: 6.3644, Disc Loss: 1.6609
Epoch 10/5000, Gen Loss: 1.6351, Disc Loss: 0.9225
Epoch 11/5000, Gen Loss: 4.5030, Disc Loss: 1.1733
Epoch 12/5000, Gen Loss: 2.1698, Disc Loss: 0.8674
Epoch 13/5000, Gen Loss: 4.8443, Disc Loss: 1.1246
Epoch 14/5000, Gen Loss: 0.8793, Disc Loss: 1.0814
Epoch 15/5000, Gen Loss: 3.7779, Disc Loss: 1.0634
Epoch 16/5000, Gen Loss: 1.3173, Disc Loss: 0.9958
Epoch 17/5000, Gen Loss: 1.4555, Disc Loss: 0.9325
Epoch 18/5000, Gen Loss: 3.0747, Disc Loss: 1.0081
Epoch 19/5000, Gen Loss: 0.8436, Disc Loss: 1.0504
Epoch 20/5000, 

In [ ]:
import os
import glob
import random
from PIL import Image
import shutil

# Diretórios
origem = 'reais_todas_combinacoes'
destino = 'Reais_Embaralhadas'
os.makedirs(destino, exist_ok=True)

# Coleta todas as imagens nos subdiretórios
arquivos = glob.glob(os.path.join(origem, '*', '*.*'))  # pega arquivos dentro de 2/, 5/, etc.
random.shuffle(arquivos)

# Copia e renomeia para o novo diretório
for i, caminho in enumerate(arquivos, start=1):
    try:
        with Image.open(caminho) as img:
            novo_nome = f"real_{i:03d}.png"
            img.save(os.path.join(destino, novo_nome), format='PNG')
    except Exception as e:
        print(f"Erro ao processar {caminho}: {e}")

print(f"{len(arquivos)} imagens embaralhadas salvas em '{destino}'.")


972 imagens embaralhadas salvas em 'Reais_Embaralhadas'.


In [ ]:
import os
import random
import shutil

# Diretórios de origem e destino
origem = 'Falsas'
destino = 'Falsas_Embaralhadas'
os.makedirs(destino, exist_ok=True)

# Lista e embaralha os arquivos
imagens = [f for f in os.listdir(origem) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
random.shuffle(imagens)

# Copia com novos nomes
for i, nome_original in enumerate(imagens, start=1):
    caminho_origem = os.path.join(origem, nome_original)
    novo_nome = f'falsa_{i:03d}.png'
    caminho_destino = os.path.join(destino, novo_nome)
    shutil.copy(caminho_origem, caminho_destino)

print(f'{len(imagens)} imagens embaralhadas e copiadas para "{destino}".')


972 imagens embaralhadas e copiadas para "Falsas_Embaralhadas".
